In [4]:
import numpy as np
#import librosa
import pandas as pd

import sys
sys.path.append("..")
from utils import UtilsIO
utils_io = UtilsIO()

In [5]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from IPython import display

# Set the seed value for experiment reproducibility.
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

2026-03-04 18:41:59.437891: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 18:41:59.440487: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 18:41:59.474459: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-04 18:41:59.474502: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-04 18:41:59.474529: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

## Load dataset

In [6]:
#df = pd._read_csv("GPS_cmd_16k_renamed_jpynb_crepe.csv")
path = "../pitch/GPS_cmd_16k_renamed_pyin_by_user.csv"
list_gps = utils_io.read_txt(path)

In [9]:
len(list_gps)

25014

In [15]:
list_gps[1]

'/mnt/c/Users/ferre/OneDrive/Documentos/Doutorado/dataset/GPS_cmd_16k_renamed/academia_F001_001.wav,F001,F,academia,"[[183.7917368 ]'

In [17]:
path = "/home/jovyan/work/OneDrive/Documentos/Doutorado/dataset/GPS_cmd_16k_renamed/academia_F001_001.wav"
test_file = tf.io.read_file(path)
test_audio, _ = tf.audio.decode_wav(contents=test_file)
test_audio.shape

TensorShape([17705, 1])

In [42]:
batch_size, num_samples, sample_rate = 32, 16000, 16000.0
# A Tensor of [batch_size, num_samples] mono PCM samples in the range [-1, 1].
#pcm = tf.random.normal([batch_size, num_samples], dtype=tf.float32)
pcm = tf.random.normal([num_samples], dtype=tf.float32)
pcm.shape

TensorShape([16000])

In [43]:
# A 1024-point STFT with frames of 64 ms and 75% overlap.
#stfts = tf.signal.stft(pcm, frame_length=1024, frame_step=256, fft_length=1024)
stfts = tf.signal.stft(pcm, frame_length=255, frame_step=128)
spectrograms = tf.abs(stfts)
spectrograms.shape

TensorShape([124, 129])

In [44]:
# Warp the linear scale spectrograms into the mel-scale.
num_spectrogram_bins = stfts.shape[-1]
print(num_spectrogram_bins)

129


In [45]:
lower_edge_hertz, upper_edge_hertz, num_mel_bins = 80.0, 7600.0, 80
linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
  num_mel_bins, num_spectrogram_bins, sample_rate, lower_edge_hertz,
  upper_edge_hertz)
print("linear_to_mel_weight_matrix: ", linear_to_mel_weight_matrix.shape)
mel_spectrograms = tf.tensordot(
  spectrograms, linear_to_mel_weight_matrix, 1)
print("mel_spectrograms: ", mel_spectrograms.shape)
print("spectrograms.shape[:-1]: ", spectrograms.shape[:-1])
mel_spectrograms.set_shape(spectrograms.shape[:-1].concatenate(
  linear_to_mel_weight_matrix.shape[-1:]))
print("mel_spectrograms: ", mel_spectrograms.shape)

# Compute a stabilized log to get log-magnitude mel-scale spectrograms.
log_mel_spectrograms = tf.math.log(mel_spectrograms + 1e-6)
print("log_mel_spectrograms: ", log_mel_spectrograms.shape)

# Compute MFCCs from log_mel_spectrograms and take the first 13.
mfccs = tf.signal.mfccs_from_log_mel_spectrograms(
  log_mel_spectrograms)[..., :13]
print(mfccs.shape)

linear_to_mel_weight_matrix:  (129, 80)
mel_spectrograms:  (124, 80)
spectrograms.shape[:-1]:  (124,)
mel_spectrograms:  (124, 80)
log_mel_spectrograms:  (124, 80)
(124, 13)


Reference: 
- https://www.tensorflow.org/tutorials/audio/simple_audio?hl=pt-br
- https://medium.com/@oluyaled/audio-classification-using-deep-learning-and-tensorflow-a-step-by-step-guide-5327467ee9ab
- https://www.tensorflow.org/api_docs/python/tf/signal/mfccs_from_log_mel_spectrograms
- https://www.tensorflow.org/api_docs/python/tf/signal/stft